In [ ]:
import h5py, numpy as np, pandas as pd

In [ ]:
# row <-> appln_id map
with h5py.File('docvecs_23_4_test.h5', 'r') as f:
    ids = f['appln_id'][:]
# appln_id: idx
id2row = {int(a): r for r, a in enumerate(ids)}

sim_f = h5py.File('similarities_23_2.h5', 'r')
sims  = sim_f['sims_50']

# query column, read once; drop the unused zero rows the resize left at the end
q = sims[:, 0].astype(np.int64)
breaks = np.flatnonzero(np.diff(q) < 0)
n_valid = breaks[0] + 1 if len(breaks) else len(q)
q = q[:n_valid]

def similar_to(appln_id, drop_self=False):
    """All neighbors stored for one appln_id, nearest first."""
    row = id2row[appln_id]
    lo, hi = np.searchsorted(q, row, 'left'), np.searchsorted(q, row, 'right')
    block = sims[lo:hi]                               # reads only this slice from disk
    nbr, dist = block[:, 1].astype(np.int64), block[:, 2]
    if drop_self:
        keep = nbr != row
        nbr, dist = nbr[keep], dist[keep]
    return pd.DataFrame({
        'appln_id':     appln_id,
        'similar_id':   ids[nbr],
        'distance':     dist,
        })

In [ ]:
similar_to(123456789) # enter appln_id